# Install NumPy

In [1]:
#!pip uninstall numpy==1.25.2
! pip uninstall numpy
#!apt-cache madison python3-numpy

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Would remove:
    /usr/local/bin/f2py
    /usr/local/bin/numpy-config
    /usr/local/lib/python3.11/dist-packages/numpy-2.0.2.dist-info/*
    /usr/local/lib/python3.11/dist-packages/numpy.libs/libgfortran-040039e1-0352e75f.so.5.0.0
    /usr/local/lib/python3.11/dist-packages/numpy.libs/libquadmath-96973f99-934c22de.so.0.0.0
    /usr/local/lib/python3.11/dist-packages/numpy.libs/libscipy_openblas64_-99b71e71.so
    /usr/local/lib/python3.11/dist-packages/numpy/*
Proceed (Y/n)? y
  Successfully uninstalled numpy-2.0.2


In [2]:
#!pip install --force-reinstall -U numpy==1.19.5
#! pip install numpy==1.23.5
! pip install numpy==1.26.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 103.3 MB/s eta 0:00:00


In [1]:
import numpy
numpy.__version__

'1.26.4'

# Install & Import dependencies

In [2]:
! pip install recbole==1.1.1 ray --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 87.9 MB/s eta 0:00:00


In [ ]:
#!pip install transformers sentence-transformers sacremoses --quiet

In [3]:
import recbole
recbole.__version__

'1.1.1'

In [5]:
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)
os.chdir('/content/drive/My Drive/Guillaume/Ichrak/')

Mounted at /content/drive


In [6]:
import pandas as pd
import numpy as np
import json
import torch
import pickle
from tqdm import tqdm
import torch
#from sentence_transformers import util
from recbole.data.interaction import Interaction
from recbole.data.dataset import Dataset
from recbole.config.configurator import Config
from recbole.quick_start import run_recbole ,load_data_and_model
from recbole.utils.utils import get_model , get_tensorboard , get_trainer,get_flops
from recbole.utils.enum_type import ModelType

In [ ]:
np.float64(3)

3.0

# Common code

## Prediction Class

In [7]:
def filter_interactions_by_user(user_id, dataset, max_interactions=100):
    user_interactions = dataset[dataset['user_id'] == user_id]
    user_interactions = user_interactions.sort_values(by='timestamp', ascending=True)
    user_interactions.reset_index(drop=True, inplace=True)
    return user_interactions

# def create_interaction(data):
#   user_id_tensor = torch.tensor(data['user_id'].values.tolist()[:1])
#   item_id_list_tensor = torch.tensor([data['item_id'].values.tolist()])
#   item_length_tensor = torch.tensor([len(data['item_id'].values.tolist())])
#   result_dict = {
#       'user_id': user_id_tensor,
#       'item_id_list': item_id_list_tensor,
#       'item_length': item_length_tensor,
#   }
#   interaction  = Interaction(result_dict)
#   return interaction
def create_interaction(data):
    user_id_tensor = torch.tensor(data['user_id'].values.tolist()[:1], dtype=torch.long)
    item_id_list_tensor = torch.tensor([data['item_id'].values.tolist()], dtype=torch.long)
    item_length_tensor = torch.tensor([len(data['item_id'].values.tolist())], dtype=torch.long)

    result_dict = {
        'user_id': user_id_tensor,
        'item_id_list': item_id_list_tensor,
        'item_length': item_length_tensor,
    }
    interaction  = Interaction(result_dict)
    return interaction
def get_top_10(interaction , model ,device, top_n = 10):
 # Move the model to the GPU
  input_inter = interaction.to(device)
  scores = model.full_sort_predict(input_inter)
  #print(scores)
  item_id_list = interaction['item_id_list'][0]
  #print(item_id_list)
  recommended_item_ids = torch.argsort(scores, descending=True)
  #print(recommended_item_ids)
  top_n_item_ids = recommended_item_ids[0][:top_n].tolist()
  return scores ,top_n_item_ids

## Metrics

In [ ]:
# precision
def calculate_precision_at_k(actual, predicted, k):
    actual_set = set(actual)
    predicted_at_k = predicted[:k]
    correct_predictions = len(actual_set.intersection(predicted_at_k))
    return correct_predictions / k  # Precision@k
# recall
def calculate_recall_at_k(actual, predicted, k):
    actual_set = set(actual)
    predicted_at_k = predicted[:k]
    correct_predictions = len(actual_set.intersection(predicted_at_k))
    return correct_predictions / len(actual_set)  # Recall@k
# Hit ratio
def calculate_hit_rate_at_k(actual, predicted, k):
    actual_set = set(actual)
    predicted_at_k = predicted[:k]
    for item in predicted_at_k:
        if item in actual_set:
            #print(item)
            return 1
    return 0
# reciprocal rank
def calculate_reciprocal_rank(actual, predicted, k):
    actual_set = set(actual)
    for i, item in enumerate(predicted[:k]):
        if item in actual_set:
            return 1.0 / (i + 1)
    return 0.0
# reciprocal rank
# def calculate_reciprocal_rank(actual, predicted, k):
#     actual_set = set(actual)
#     mrr = 0
#     for i, item in enumerate(predicted[:k]):
#         if item in actual_set:
#             x= 1.0 / (i + 1)
#             mrr += x
#             return mrr/k
#     return 0.0
#ndcg
def calculate_dcg_at_k(actual, predicted, k):
    actual_set = set(actual)
    dcg = 0
    for i, item in enumerate(predicted[:k]):
        if item in actual_set:
            relevance = 1
        else:
            relevance = 0
        dcg += (2 ** relevance - 1) / np.log2(i + 2)
    return dcg

def calculate_ndcg_at_k(actual, predicted, k):
    idcg = calculate_dcg_at_k(actual, actual, k)
    dcg = calculate_dcg_at_k(actual, predicted, k)
    return dcg / idcg if idcg > 0 else 0

## Train Model (BERT4REC)

In [ ]:
# import time
# while True:
#     print("Keeping session alive...")
#     time.sleep(120)

In [ ]:
config_dict = {
    'data_path': './Coursera_data',
    'dataset': 'courses_data_recbole',
    'USER_ID_FIELD': 'user_id',
    'ITEM_ID_FIELD': 'item_id',
    'RATING_FIELD': 'rating',
    #'seq_separator' : '[SEP]',
    'TIME_FIELD': 'timestamp',
    'load_col': {
        'inter': ['user_id', 'item_id', 'rating', 'timestamp']
    } ,
    'loss_type': 'CE',
    'train_neg_sample_args': None,
    'eval_args':{
       'split': {'RS': [9,0,1]},
    'group_by': 'user',
    'order': 'TO',
    'mode': 'full'
    },
    #'train_batch_size' : 256 ,
    'learning_rate': 0.0001 ,
    'weight_decay' : 0.01 ,
    #'eval_batch_size' : 256 ,
    'mask_ratio':0.1 ,
    #'MAX_ITEM_LIST_LENGTH': 200


}
run_recbole(model='BERT4Rec', dataset='courses_data_recbole', config_dict=config_dict)

/usr/local/lib/python3.11/dist-packages/recbole/data/dataset/dataset.py:638: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
/usr/local/lib/python3.11/dist-packages/recbole/data/dataset/dataset.py:640: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a c

{'best_valid_score': -inf,
 'valid_score_bigger': True,
 'best_valid_result': None,
 'test_result': OrderedDict([('recall@10', 0.3876),
              ('mrr@10', 0.178),
              ('ndcg@10', 0.2266),
              ('hit@10', 0.3876),
              ('precision@10', 0.0388)])}

## Train Model (SASRec)

In [ ]:
config_dict = {
    'data_path': './movies_data',
    'dataset': 'ratings_genres_recbole',
    'USER_ID_FIELD': 'user_id',
    'ITEM_ID_FIELD': 'item_id',
    'RATING_FIELD': 'rating',
    #'seq_separator' : '[SEP]',
    'TIME_FIELD': 'timestamp',
    'load_col': {
        'inter': ['user_id', 'item_id', 'rating', 'timestamp']
    } ,
    'loss_type': 'CE',
    'train_neg_sample_args': None,
    'eval_args':{
       'split': {'RS': [7,2,1]},
    'group_by': 'user',
    'order': 'TO',
    'mode': 'full'
    },
    #'train_batch_size' : 256 ,
    'learning_rate': 0.0001 ,
    'weight_decay' : 0.01 ,
    #'eval_batch_size' : 256 ,
    'mask_ratio':0.2 ,
    #'MAX_ITEM_LIST_LENGTH': 200


}
run_recbole(model='SASRec', dataset='ratings_genres_recbole', config_dict=config_dict)

Train     0:   0%|                                                           | 0/92 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/recbole/trainer/trainer.py:236: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)
Evaluate   : 100%|█████████████████████████| 13/13 [00:00<00:00, 13.22it/s, GPU RAM: 2.81 G/14.75 G]
/usr/local/lib/python3.10/dist-packages/recbole/trainer/trainer.py:583: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be exe

{'best_valid_score': 0.5592,
 'valid_score_bigger': True,
 'best_valid_result': OrderedDict([('recall@10', 0.9087),
              ('mrr@10', 0.5592),
              ('ndcg@10', 0.6428),
              ('hit@10', 0.9087),
              ('precision@10', 0.0909)]),
 'test_result': OrderedDict([('recall@10', 0.9215),
              ('mrr@10', 0.5814),
              ('ndcg@10', 0.6628),
              ('hit@10', 0.9215),
              ('precision@10', 0.0922)])}

#Processing data for evaluation

In [ ]:
test_data = pd.read_csv('data/amazon_movies_categories_final/amazon_movies_categories_with_sep_test_data_final.csv')

In [ ]:
test_data.shape

(8176399, 7)

In [ ]:
test_data.isna().sum()

user_id              0
item_id              0
rating               0
timestamp            0
category           250
title               77
category_mapped      0
dtype: int64

In [ ]:
#keep users inter between 5 and 200
#test_data.head(10)
# Assuming 'user_id' is the column containing user IDs and 'item_id' is the column containing item IDs
sequence_lengths = test_data.groupby('user_id')['item_id'].count()
# Print the sequence lengths
#print(sequence_lengths)
# If you want to get the average sequence length
average_sequence_length = sequence_lengths.mean()
print(f"Average sequence length: {average_sequence_length}")
print(sequence_lengths.min())
print(sequence_lengths.max())

Average sequence length: 19.550987664062706
2
17535


In [ ]:
# delete nan values and keep users that have alredy 5 interaction
test_data = test_data.dropna()
# Assuming 'user_id' is the column containing user IDs and 'item_id' is the column containing item IDs
user_interaction_counts = test_data['user_id'].value_counts()
# Filter users with sequences between 5 and 200 interactions
valid_users = user_interaction_counts[(user_interaction_counts >= 5) & (user_interaction_counts <= 200)].index
test_data = test_data[test_data['user_id'].isin(valid_users)]


In [ ]:
test_data.shape

(7205645, 7)

In [ ]:
test_data.head(11)

,user_id,item_id,rating,timestamp,category,title,category_mapped
0,A3M3HCZLXW0YLF,0001527665,5.0,1342310400,Movies & TV,Peace Child VHS,CAT_1
1,A3M3HCZLXW0YLF,0001527665,5.0,1342320400,Art House & International,Peace Child VHS,CAT_2
2,A3M3HCZLXW0YLF,0001527665,5.0,1342330400,By Original Language,Peace Child VHS,CAT_3
3,A3M3HCZLXW0YLF,0001527665,5.0,1342340400,Spanish,Peace Child VHS,CAT_4
4,A3M3HCZLXW0YLF,0001527665,5.0,1342350400,[SEP],Peace Child VHS,CAT_0
5,A16WO8T4YXGVWP,0005089549,5.0,1277596800,Movies & TV,Cathedral Quartet: A Reunion VHS,CAT_1
6,A16WO8T4YXGVWP,0005089549,5.0,1277606800,Genre for Featured Categories,Cathedral Quartet: A Reunion VHS,CAT_5
7,A16WO8T4YXGVWP,0005089549,5.0,1277616800,Faith & Spirituality,Cathedral Quartet: A Reunion VHS,CAT_6
8,A16WO8T4YXGVWP,0005089549,5.0,1277626800,[SEP],Cathedral Quartet: A Reunion VHS,CAT_0
9,A2A4GWAEM3VOW0,000503860X,5.0,1116374400,Movies & TV,Chapter X Live [VHS],CAT_1


In [ ]:
selected_users = ["A3M3HCZLXW0YLF", "A16WO8T4YXGVWP"]
df = test_data[test_data['user_id'].isin(selected_users)]
df.shape

(27, 7)

In [ ]:

# Sort the dataframe by 'timestamp' within each user-item group
grouped_by_user = test_data.sort_values(by='timestamp', ascending=True).groupby(['user_id','item_id']).agg(list).reset_index()
grouped_by_user
last_rows = grouped_by_user.groupby('user_id').tail(1)
last_rows
new_dataset = last_rows[['user_id', 'category', 'category_mapped']].copy()
new_dataset
test_data = test_data[~test_data.set_index(['user_id', 'item_id']).index.isin(last_rows.set_index(['user_id', 'item_id']).index)].copy()
test_data.shape
# Group by 'user_id' and 'item_id' and get the last interaction for each user-item combination
# df_grouped = df_sorted.groupby(['user_id', 'item_id']).apply(lambda group: group.iloc[0]).reset_index(drop=True)
# df_grouped
# Create a new DataFrame with 'user_id' and 'actual_sequence' (list of categories for the last item)
#new_dataset = df_grouped.groupby('user_id')['category_mapped'].agg(list).reset_index(name='actual_sequence')

# Drop the last interactions from the original dataset
#df = df[~df.set_index(['user_id', 'item_id']).index.isin(df_grouped.set_index(['user_id', 'item_id']).index)].copy()


(5321001, 7)

In [ ]:
# save test data
test_data.to_csv('amazon_movies_categories_evaluation_data.csv', index=False)
# sava new dataset
new_dataset.to_csv('amazon_movies_categories_last_actual_seq.csv', index=False)


# Evaluate Model with SEP token

In [ ]:
# important notes


### Load model (BERT4REC)

In [ ]:
#load saved model  (with sep  )
#movies BERT4REC  "saved/BERT4Rec-Oct-02-2024_07-22-23.pth"
#courses BERT4REC "BERT4Rec-Mar-05-2025_12-31-20.pth"
# courses BERT4REC ""
config, model, dataset, train_data, valid_data, test_data = load_data_and_model(
    model_file='saved/BERT4Rec-Mar-05-2025_12-31-20.pth', weights_only=False
)


TypeError: load_data_and_model() got an unexpected keyword argument 'weights_only'

In [8]:
# Patch temporaire de torch.load
_original_load = torch.load

def patched_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_load(*args, **kwargs)

torch.load = patched_load  # Patch appliqué

# Appel à RecBole avec le fichier .pth
config, model, dataset, train_data, valid_data, test_data = load_data_and_model(
    model_file='saved/BERT4Rec-Mar-05-2025_12-31-20.pth',
)

# Optionnel : restaurer torch.load si tu veux être propre
torch.load = _original_load

/usr/local/lib/python3.11/dist-packages/recbole/data/dataset/dataset.py:638: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
/usr/local/lib/python3.11/dist-packages/recbole/data/dataset/dataset.py:640: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a c

In [ ]:
dataset

courses_data_recbole
The number of users: 45501
Average actions of users: 42.0875671392093
The number of items: 153
Average actions of items: 12661.94701986755
The number of inters: 1911954
The sparsity of the dataset: 72.53591927089731%
Remain Fields: ['user_id', 'item_id', 'rating', 'timestamp', 'item_id_list', 'rating_list', 'timestamp_list', 'item_length']

In [ ]:
#config

### Load Model (SASRec)

In [ ]:
#load saved model  (with sep  )
config, model, dataset, train_data, valid_data, test_data = load_data_and_model(
    model_file='saved/SASRec-Sep-18-2024_11-44-23.pth',
)


/usr/local/lib/python3.10/dist-packages/recbole/quick_start/quick_start.py:178: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_file)


In [ ]:
#config

### Get predictions top 10

In [ ]:
os.getcwd()

'/content/drive/.shortcut-targets-by-id/1ONhraksKqsrnlDQCMyVrfQGAIMQlJnxg/Guillaume/Ichrak'

In [ ]:
import pandas as pd

In [9]:
data = pd.read_csv("Coursera_data/courses_categories_with_sep_test_data.csv")
data.head()

,User ID,User Name,Course Name,Timestamp,Rating,Course Rating,categories,categories_mapped
0,ca386a74,Mary Griffin,create a financial statement using google sheets,2024-06-12 09:52:14.282690,5.0,4.8,"Algebra, Arithmetic, and Discrete Mathematics",CAT_54
1,ca386a74,Mary Griffin,create a financial statement using google sheets,2024-06-12 09:52:24.282690,5.0,4.8,Data & Statistical Analysis,CAT_55
2,ca386a74,Mary Griffin,create a financial statement using google sheets,2024-06-12 09:52:34.282690,5.0,4.8,Virtual & Augmented Reality,CAT_15
3,ca386a74,Mary Griffin,create a financial statement using google sheets,2024-06-12 09:52:44.282690,5.0,4.8,Accounting & Auditing,CAT_58
4,ca386a74,Mary Griffin,create a financial statement using google sheets,2024-06-12 09:52:54.282690,5.0,4.8,Cost Management & Corporate Finance,CAT_31


In [ ]:
len(data['User ID'].unique())

19500

In [ ]:
genres_dict = dict(zip(data['categories_mapped'], data['categories']))
genres_dict

{'CAT_54': 'Algebra, Arithmetic, and Discrete Mathematics',
 'CAT_55': 'Data & Statistical Analysis',
 'CAT_15': 'Virtual & Augmented Reality',
 'CAT_58': 'Accounting & Auditing',
 'CAT_31': 'Cost Management & Corporate Finance',
 'CAT_59': 'Banking & Financial Transactions',
 'CAT_3': 'Operations, Risk Management & Logistics',
 'CAT_60': 'Banking & Financial Institutions',
 'CAT_61': 'General Finance & Investment',
 'CAT_0': '[SEP]',
 'CAT_62': 'Investment & Portfolio Management',
 'CAT_63': 'Computer Networking & Network Infrastructure',
 'CAT_38': 'Psychology & Behavioral Science',
 'CAT_64': 'Entrepreneurial & Corporate Finance',
 'CAT_65': 'Engineering & Technology Leadership',
 'CAT_37': 'Management & Operations',
 'CAT_36': 'Economic Analysis & Market Dynamics',
 'CAT_45': 'Strategic Planning & Policy Making',
 'CAT_75': 'Marketing & Market Strategy',
 'CAT_76': 'Business Development & Business Intelligence',
 'CAT_43': 'Human Resources & Organizational Development',
 'CAT_44': 

In [ ]:
pickle_file_path = 'courses_categories_mapping.pkl'
with open(pickle_file_path, 'wb') as file:
    pickle.dump(genres_dict, file)

print(f"Dictionary saved to {pickle_file_path}")


Dictionary saved to courses_categories_mapping.pkl


In [10]:
selected_columns = ['User ID', 'categories_mapped', 'Rating', 'Timestamp']
data = data[selected_columns]

In [11]:
column_names = ['user_id', 'item_id', 'rating', 'timestamp']
data.columns = column_names

In [ ]:
data.shape

(838729, 4)

In [ ]:
grouped_data = data.groupby('user_id').agg(list).reset_index()
grouped_data.head()

,user_id,item_id,rating,timestamp
0,00061623,"[CAT_70, CAT_60, CAT_31, CAT_55, CAT_23, CAT_5...","[3.2, 3.2, 3.2, 3.2, 3.2, 3.2, 3.2, 3.2, 3.2, ...","[2024-04-14 09:54:28.888989, 2024-04-14 09:54:..."
1,0006b8f2,"[CAT_76, CAT_45, CAT_82, CAT_3, CAT_32, CAT_11...","[3.4, 3.4, 3.4, 3.4, 3.4, 3.4, 3.4, 3.4, 3.4, ...","[2024-08-09 10:24:39.982693, 2024-08-09 10:24:..."
2,0007df43,"[CAT_29, CAT_56, CAT_34, CAT_52, CAT_65, CAT_5...","[2.2, 2.2, 2.2, 2.2, 2.2, 2.2, 2.2, 2.2, 2.2, ...","[2024-02-22 10:31:50.502517, 2024-02-22 10:32:..."
3,000876d6,"[CAT_8, CAT_54, CAT_74, CAT_66, CAT_14, CAT_19...","[4.6, 4.6, 4.6, 4.6, 4.6, 4.6, 4.6, 4.6, 4.6, ...","[2024-10-28 09:53:19.730988, 2024-10-28 09:53:..."
4,000a6d4c,"[CAT_35, CAT_43, CAT_26, CAT_3, CAT_77, CAT_87...","[4.5, 4.5, 4.5, 4.5, 4.5, 4.5, 4.5, 4.5, 5.0, ...","[2024-12-23 10:00:26.693894, 2024-12-23 10:00:..."


In [ ]:
grouped_data['item_id'].iloc[0]

['CAT_70',
 'CAT_60',
 'CAT_31',
 'CAT_55',
 'CAT_23',
 'CAT_59',
 'CAT_3',
 'CAT_61',
 'CAT_0',
 'CAT_59',
 'CAT_31',
 'CAT_3',
 'CAT_10',
 'CAT_60',
 'CAT_54',
 'CAT_102',
 'CAT_45',
 'CAT_83',
 'CAT_0',
 'CAT_56',
 'CAT_19',
 'CAT_3',
 'CAT_58',
 'CAT_55',
 'CAT_43',
 'CAT_10',
 'CAT_40',
 'CAT_0',
 'CAT_6',
 'CAT_19',
 'CAT_54',
 'CAT_56',
 'CAT_28',
 'CAT_55',
 'CAT_10',
 'CAT_0',
 'CAT_56',
 'CAT_6',
 'CAT_55',
 'CAT_54',
 'CAT_19',
 'CAT_7',
 'CAT_10',
 'CAT_40',
 'CAT_0']

In [ ]:
grouped_data.columns

Index(['user_id', 'item_id', 'rating', 'timestamp'], dtype='object')

In [ ]:
# Function to remove the last interaction sequence
def process_interactions(item_ids, ratings, timestamps):
    # Find indices of 'CAT_0'
    cat_0_indices = [i for i, x in enumerate(item_ids) if x == 'CAT_0']

    if len(cat_0_indices) >= 2:
        # Get the index of the second-to-last 'CAT_0'
        split_index = cat_0_indices[-2]

        # Keep parts of the lists up to and including the second-to-last 'CAT_0'
        return (
            item_ids[:split_index + 1],  # Modify item_id list
            ratings[:split_index + 1],    # Modify ratings list
            timestamps[:split_index + 1]   # Modify timestamp list
        )
    else:
        # If fewer than 2 'CAT_0's, return the full lists
        return (item_ids, ratings, timestamps)

# Apply the processing function to the grouped data
grouped_data[['item_id', 'rating', 'timestamp']] = grouped_data.apply(
    lambda row: pd.Series(process_interactions(
        row['item_id'], row['rating'], row['timestamp']
    )),
    axis=1
)

# # Explode the list columns back into individual rows (ungroup)
new_data = grouped_data.explode(['item_id', 'rating', 'timestamp'])

# Reset index to make the DataFrame flat again
new_data = new_data.reset_index(drop=True)

# Debugging - Print the shape of the exploded data to verify the change
print("Shape of exploded data:", new_data.shape)
print("Exploded data after removing the last sequence:")
new_data.head()

Shape of exploded data: (671061, 4)
Exploded data after removing the last sequence:


,user_id,item_id,rating,timestamp
0,00061623,CAT_70,3.2,2024-04-14 09:54:28.888989
1,00061623,CAT_60,3.2,2024-04-14 09:54:38.888989
2,00061623,CAT_31,3.2,2024-04-14 09:54:48.888989
3,00061623,CAT_55,3.2,2024-04-14 09:54:58.888989
4,00061623,CAT_23,3.2,2024-04-14 09:55:08.888989


In [ ]:
new_data.to_csv('courses_categories_evaluation_data_without_last_inter.csv', index=False)

In [12]:
new_data = pd.read_csv('Coursera_data/courses_categories_evaluation_data_without_last_inter.csv')

In [ ]:
def process_item_list(item_list):
    cat_0_indices = [i for i, x in enumerate(item_list) if x == 'CAT_0']
    if len(cat_0_indices) >= 2:
        split_index = cat_0_indices[-2]
        last_interaction = item_list[split_index + 1 :]
        modified_list = item_list[:split_index + 1]
        return modified_list, last_interaction
    else:
        return item_list, []
grouped_data[['modified_list', 'last_interaction']] = grouped_data['item_id'].apply(lambda x: pd.Series(process_item_list(x)))
last_interaction_df = grouped_data[['user_id', 'last_interaction']]

In [ ]:
last_interaction_df

,user_id,last_interaction
0,00061623,"[CAT_6, CAT_19, CAT_54, CAT_56, CAT_28, CAT_55..."
1,0006b8f2,"[CAT_79, CAT_43, CAT_37, CAT_3, CAT_33, CAT_44..."
2,0007df43,"[CAT_58, CAT_7, CAT_66, CAT_68, CAT_19, CAT_69..."
3,000876d6,"[CAT_3, CAT_33, CAT_37, CAT_31, CAT_25, CAT_45..."
4,000a6d4c,"[CAT_26, CAT_1, CAT_3, CAT_28, CAT_37, CAT_75,..."
...,...,...
19495,fff5a572,"[CAT_92, CAT_63, CAT_8, CAT_88, CAT_14, CAT_66..."
19496,fff66b43,"[CAT_26, CAT_132, CAT_1, CAT_129, CAT_74, CAT_..."
19497,fffa0c5f,"[CAT_35, CAT_43, CAT_26, CAT_3, CAT_77, CAT_87..."
19498,fffd5106,"[CAT_55, CAT_1, CAT_7, CAT_48, CAT_56, CAT_20,..."


In [ ]:
last_interaction_df.to_csv('courses_categories_last_actual_seq.csv', index=False)

In [13]:
items_ids = dataset.field2token_id['item_id']
users_ids = dataset.field2token_id['user_id']

In [ ]:
items_ids

{'[PAD]': 0,
 'CAT_1': 1,
 'CAT_2': 2,
 'CAT_3': 3,
 'CAT_4': 4,
 'CAT_5': 5,
 'CAT_0': 6,
 'CAT_6': 7,
 'CAT_7': 8,
 'CAT_8': 9,
 'CAT_9': 10,
 'CAT_10': 11,
 'CAT_11': 12,
 'CAT_12': 13,
 'CAT_13': 14,
 'CAT_14': 15,
 'CAT_15': 16,
 'CAT_16': 17,
 'CAT_17': 18,
 'CAT_18': 19,
 'CAT_19': 20,
 'CAT_20': 21,
 'CAT_21': 22,
 'CAT_22': 23,
 'CAT_23': 24,
 'CAT_24': 25,
 'CAT_25': 26,
 'CAT_26': 27,
 'CAT_27': 28,
 'CAT_28': 29,
 'CAT_29': 30,
 'CAT_30': 31,
 'CAT_31': 32,
 'CAT_32': 33,
 'CAT_33': 34,
 'CAT_34': 35,
 'CAT_35': 36,
 'CAT_36': 37,
 'CAT_37': 38,
 'CAT_38': 39,
 'CAT_39': 40,
 'CAT_40': 41,
 'CAT_41': 42,
 'CAT_42': 43,
 'CAT_43': 44,
 'CAT_44': 45,
 'CAT_45': 46,
 'CAT_46': 47,
 'CAT_47': 48,
 'CAT_48': 49,
 'CAT_49': 50,
 'CAT_50': 51,
 'CAT_51': 52,
 'CAT_52': 53,
 'CAT_53': 54,
 'CAT_54': 55,
 'CAT_55': 56,
 'CAT_56': 57,
 'CAT_57': 58,
 'CAT_66': 59,
 'CAT_67': 60,
 'CAT_68': 61,
 'CAT_69': 62,
 'CAT_70': 63,
 'CAT_60': 64,
 'CAT_71': 65,
 'CAT_72': 66,
 'CAT_73': 67,
 

In [ ]:
max_value = max(users_ids.values())
print("Maximum value in the dictionary:", max_value)

Maximum value in the dictionary: 45500


In [ ]:
new_data

,user_id,item_id,rating,timestamp
0,00061623,CAT_70,3.2,2024-04-14 09:54:28.888989
1,00061623,CAT_60,3.2,2024-04-14 09:54:38.888989
2,00061623,CAT_31,3.2,2024-04-14 09:54:48.888989
3,00061623,CAT_55,3.2,2024-04-14 09:54:58.888989
4,00061623,CAT_23,3.2,2024-04-14 09:55:08.888989
...,...,...,...,...
671056,ffff5223,CAT_25,2.4,2025-03-03 10:46:11.923909
671057,ffff5223,CAT_35,2.4,2025-03-03 10:46:21.923909
671058,ffff5223,CAT_32,2.4,2025-03-03 10:46:31.923909
671059,ffff5223,CAT_89,2.4,2025-03-03 10:46:41.923909


In [ ]:

df_inter = filter_interactions_by_user("00061623",new_data)
df_inter
# # len(df_inter['item_id'].values)
df_inter['item_id'].values.tolist()
user_id = df_inter['user_id'].values.tolist()[1]
user_id = max_value +1
#df_inter['user_id'].values.tolist()[1]
#torch.tensor(user_id)
user_id_tensor = torch.tensor(user_id)
df_inter
# #user_id_tensor
item_id_list_tensor = torch.tensor([df_inter['recbole_id'].values.tolist()])
# item_length_tensor = torch.tensor([len(df_inter['recbole_id'].values.tolist())])

# result_dict = {
#     'user_id': user_id_tensor,
#     'item_id_list': item_id_list_tensor,
#     'item_length': item_length_tensor,
# }
# result_dict
# interaction  = Interaction(result_dict)
# interaction
# input_inter = interaction.to(device)
# input_inter
# # formatted_data = interaction_to_dict(df_inter)
# # formatted_data
# # dataa = pd.DataFrame(formatted_data)
# # dataa
# # interaction = Interaction(dataa)
# # interaction
# predicted , scores  = get_top_10(input_inter , model ,device, top_n = 10)
# scores

,user_id,item_id,rating,timestamp
0,00061623,CAT_70,3.2,2024-04-14 09:54:28.888989
1,00061623,CAT_60,3.2,2024-04-14 09:54:38.888989
2,00061623,CAT_31,3.2,2024-04-14 09:54:48.888989
3,00061623,CAT_55,3.2,2024-04-14 09:54:58.888989
4,00061623,CAT_23,3.2,2024-04-14 09:55:08.888989
5,00061623,CAT_59,3.2,2024-04-14 09:55:18.888989
6,00061623,CAT_3,3.2,2024-04-14 09:55:28.888989
7,00061623,CAT_61,3.2,2024-04-14 09:55:38.888989
8,00061623,CAT_0,3.2,2024-04-14 09:55:48.888989
9,00061623,CAT_59,3.0,2024-04-27 09:55:58.888989


In [ ]:
# input_inter = Interaction({
#         'user_id': torch.tensor([1]),
#         'item_id_list': torch.tensor([[1, 12, 20, 21, 0, 1, 77, 78, 0]]),
#         'item_length': torch.tensor([9]),
#     })

In [ ]:
#test mapping function
user_id = '00061623'
user_interactions = data[data['user_id'] == user_id]
user_interactions = user_interactions.tail(50)
user_interactions.reset_index(drop=True, inplace=True)
mapped_user_interactions = map_user_item_ids(user_interactions, users_ids, items_ids)
mapped_user_interactions

,user_id,item_id,rating,timestamp
0,45501,63,3.2,2024-04-14 09:54:28.888989
1,45501,64,3.2,2024-04-14 09:54:38.888989
2,45501,32,3.2,2024-04-14 09:54:48.888989
3,45501,56,3.2,2024-04-14 09:54:58.888989
4,45501,24,3.2,2024-04-14 09:55:08.888989
5,45501,74,3.2,2024-04-14 09:55:18.888989
6,45501,3,3.2,2024-04-14 09:55:28.888989
7,45501,144,3.2,2024-04-14 09:55:38.888989
8,45501,6,3.2,2024-04-14 09:55:48.888989
9,45501,74,3.0,2024-04-27 09:55:58.888989


In [ ]:
%%time
import random
device = torch.device('cuda:0')
model = model.to(device)
user_id = '00061623'
sep_token_id = items_ids['CAT_0']  # Assuming 'CAT_0' is the key for the separator token in items_dict
predicted_sequence ,scores= predict_and_add_until_sep(user_id, data, model,device, items_ids, users_ids, sep_token_id, top_n=10)
print(predicted_sequence)

<ipython-input-23-f8ffc98a070b>:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  user_interactions['timestamp'] = pd.to_datetime(user_interactions['timestamp'])


[56, 56, 6]
CPU times: user 1.02 s, sys: 230 ms, total: 1.25 s
Wall time: 1.78 s


In [ ]:
len(items_ids)

153

In [ ]:
# unknown_items = new_data[~new_data['item_id'].isin(items_ids)].item_id.unique()
# if len(unknown_items) > 0:
#     print(f"[User {user_id}] unknown items:", unknown_items)
print("Max index in items_dict:", max(items_ids.values()))
print("Max item index in user_data:", new_data['item_id'].map(items_ids).max())



Max index in items_dict: 152
Max item index in user_data: 152


In [ ]:
new_data['timestamp'] = pd.to_datetime(new_data['timestamp'])
new_data['Timestamp'] = new_data['timestamp'].astype('int64') / 1e9  # Convert nanoseconds to seconds


In [14]:
import torch
import random
import os
import pandas as pd
import pickle
from tqdm import tqdm
import warnings

# Removed the fixed seed to make the process non-deterministic
device = torch.device('cuda:0')
model = model.to(device)

# Function definitions
# def map_user_item_ids(user_interactions, users_dict, items_dict):
#     mapped_user_interactions = user_interactions.copy()
#     mapped_user_interactions['user_id'] = mapped_user_interactions['user_id'].apply(lambda x: users_dict.get(x, max(users_dict.values()) + 1))
#     mapped_user_interactions['item_id'] = mapped_user_interactions['item_id'].apply(lambda x: items_dict.get(x, max(items_dict.values()) + 1))
#     return mapped_user_interactions
def map_user_item_ids(user_interactions, users_dict, items_dict):
    known_items = set(items_dict.keys())
    known_users = set(users_dict.keys())

    # Filtrer les inconnus
    filtered = user_interactions[
        user_interactions['item_id'].isin(known_items) & user_interactions['user_id'].isin(known_users)
    ].copy()

    # if len(filtered) < len(user_interactions):
    #     print(f"[Warning] Ignored {len(user_interactions) - len(filtered)} unknown user/item interactions.")

    # Mapper les IDs
    filtered['user_id'] = filtered['user_id'].map(users_dict)
    filtered['item_id'] = filtered['item_id'].map(items_dict)

    return filtered
def predict_next_item(interaction, model, device, top_n=33):
    interaction = create_interaction(interaction)
    scores, predicted_top_n = get_top_10(interaction, model, device, top_n=top_n)
    return predicted_top_n[0], scores

def add_interaction(user_interactions, user_id, predicted_item_id, timestamp_increment=10000):
    user_interactions['timestamp'] = pd.to_datetime(user_interactions['timestamp'])
    new_timestamp = user_interactions['timestamp'].iloc[-1] + pd.to_timedelta(timestamp_increment, unit='ms')
    new_interaction = pd.DataFrame({
        'user_id': [user_id],
        'item_id': ["CAT_" + str(predicted_item_id)],
        'rating': [5],
        'timestamp': [new_timestamp]
    })
    return pd.concat([user_interactions, new_interaction], ignore_index=True)

def predict_and_add_until_sep(user_id, dataset, model, device, items_dict, users_dict, sep_token_id, top_n=33):
    user_interactions = dataset[dataset['user_id'] == user_id]
    if user_interactions.empty:
        return [], []

    predicted_sequence = []
    scores_matrix = []
    max_predictions = 33

    retries = 0  # Retry counter
    retry_limit = 20  # Default retry limit

    while len(predicted_sequence) < max_predictions:
        # Map user and item ids to model format
        mapped_user_interactions = map_user_item_ids(user_interactions, users_dict, items_dict)
        predicted_item_id, scores = predict_next_item(mapped_user_interactions, model, device, top_n=top_n)

        # Retry logic: Increase the retry limit to 30 if 'CAT_0' (id 6) is predicted in the first two positions
        if len(predicted_sequence) < 2 and predicted_item_id == 6:
            retries += 1
            retry_limit = 30  # Set retry limit to 30 if CAT_0 is predicted early in the sequence
            if retries >= retry_limit:
                break  # Accept 'CAT_0' after hitting the retry limit
            continue  # Retry if 'CAT_0' is predicted in the first two positions

        # If we get a valid item, append it
        predicted_sequence.append(predicted_item_id)
        scores_matrix.append(scores)

        # Reset retries after successful predictions
        retries = 0
        retry_limit = 6  # Reset retry limit back to 6 for other cases

        if predicted_item_id == sep_token_id:
            break

        # Add predicted item to user interactions
        user_interactions = add_interaction(user_interactions, user_id, predicted_item_id)

    # Return the predicted sequence and corresponding scores
    return predicted_sequence, scores_matrix
# Checkpoint save/load functions
def load_processed_batches(output_directory_sequences):
    processed_batches = set()
    if os.path.exists(output_directory_sequences):
        for file_name in os.listdir(output_directory_sequences):
            if file_name.startswith('batch_') and file_name.endswith('.csv'):
                batch_num = int(file_name.split('_')[1].split('.')[0])
                processed_batches.add(batch_num)
    return processed_batches

def save_checkpoint(result_df, batch_scores_dict, batch_num, output_directory_sequences, output_directory_scores):
    # Save the batch sequences to a CSV file
    batch_sequences_filename = f'{output_directory_sequences}batch_{batch_num}.csv'
    result_df.to_csv(batch_sequences_filename, index=False)

    # Save the batch scores to a pickle file
    batch_scores_filename = f'{output_directory_scores}batch_{batch_num}.pickle'
    with open(batch_scores_filename, 'wb') as f:
        pickle.dump(batch_scores_dict, f)


In [ ]:
new_data.head()

,user_id,item_id,rating,timestamp,Timestamp
0,00061623,CAT_70,3.2,2024-04-14 09:54:28.888989,1.713088e+09
1,00061623,CAT_60,3.2,2024-04-14 09:54:38.888989,1.713088e+09
2,00061623,CAT_31,3.2,2024-04-14 09:54:48.888989,1.713088e+09
3,00061623,CAT_55,3.2,2024-04-14 09:54:58.888989,1.713088e+09
4,00061623,CAT_23,3.2,2024-04-14 09:55:08.888989,1.713089e+09


In [ ]:
warnings.simplefilter(action='ignore', category=FutureWarning)

# Output directories for saving intermediate results
output_directory_sequences = 'batch_results_sequences_Bert4rec_last_version/'
output_directory_scores = 'batch_results_scores_Bert4rec_last_version/'
os.makedirs(output_directory_sequences, exist_ok=True)
os.makedirs(output_directory_scores, exist_ok=True)

result_columns = ['user_id', 'predicted_sequence']
result_df = pd.DataFrame(columns=result_columns)
batch_size = 100

# Get unique users in batches
unique_users = new_data['user_id'].unique()
total_batches = len(unique_users) // batch_size + int(len(unique_users) % batch_size > 0)

# Load already processed batches from the output directory (for checkpointing)
processed_batches = load_processed_batches(output_directory_sequences)

# Process batches with checkpoints
for batch_num in tqdm(range(total_batches)):
    # Skip already processed batches
    if batch_num in processed_batches:
        continue

    start_index = batch_num * batch_size
    end_index = (batch_num + 1) * batch_size
    batch_user_ids = unique_users[start_index:end_index]

    batch_scores_dict = {}  # Dictionary to store scores for each user in the batch
    for user_id in batch_user_ids:
        user_data = new_data[new_data['user_id'] == user_id].sort_values(by='timestamp', ascending=False)

        sep_token_id = items_ids['CAT_0']

        predicted_sequence, scores = predict_and_add_until_sep(user_id, user_data, model, device, items_ids, users_ids, sep_token_id, top_n=33)

        result_df = pd.concat([result_df, pd.DataFrame([{'user_id': user_id, 'predicted_sequence': predicted_sequence}])], ignore_index=True)

        # Store scores for the user
        batch_scores_dict[user_id] = scores

    # Save checkpoint after processing each batch
    save_checkpoint(result_df, batch_scores_dict, batch_num + 1, output_directory_sequences, output_directory_scores)

    # Clear the result DataFrame for the next batch
    result_df = pd.DataFrame(columns=result_columns)


100%|██████████| 195/195 [12:28:40<00:00, 230.36s/it]


In [ ]:
result_df

,user_id,predicted_sequence


### code exploration (to undrestand )

[https://recbole.io/docs/_modules/recbole/model/sequential_recommender/bert4rec.html#BERT4Rec.predict](https://)

In [ ]:
interaction['item_id_list']

In [ ]:
device = next(model.parameters()).device
item_seq = interaction['item_id_list'].to(device)
seq_output = model.forward(item_seq)
seq_output.shape

In [ ]:
#item_seq_len = interaction[self.ITEM_SEQ_LEN]
item_seq_len = interaction[model.ITEM_SEQ_LEN]
item_seq_len = item_seq_len.to(device)
#x = item_seq_len - 1
seq_output= model.gather_indexes(seq_output, item_seq_len - 1)
seq_output.shape

In [ ]:
 output_bias = torch.nn.Parameter(torch.zeros(model.n_items))
 output_bias = output_bias.to(device)
 output_bias.shape

In [ ]:
# test_items_emb = self.item_embedding.weight[
#             : self.n_items
#         ]  # delete masked token
#         scores = (
#             torch.matmul(seq_output, test_items_emb.transpose(0, 1)) + self.output_bias
#         )  # [B, item_num]
#model.item_embedding.weight
test_items_emb = model.item_embedding.weight[:model.n_items]
test_items_emb = test_items_emb.to(device)
scores = (torch.matmul(seq_output, test_items_emb.transpose(0, 1)) + output_bias)
#model.output_bias
#test_items_emb.transpose(0, 1)
#len(model.item_embedding.weight)

In [ ]:
t = torch.matmul(seq_output, test_items_emb.transpose(0, 1))
t.shape

In [ ]:
t

In [ ]:
e = t + output_bias
e.shape

In [ ]:
e

In [ ]:
y =test_items_emb.transpose(0, 1)
y.shape

In [ ]:
x = model.item_embedding.weight[:model.n_items]
x.shape

# Test

In [ ]:
directory = 'batch_results_scores_Bert4rec'
loaded_data = []
for filename in os.listdir(directory):
    if filename.endswith(".pickle"):
        # Construct the full path to the pickle file
        filepath = os.path.join(directory, filename)
        # Load the data from the pickle file
        with open(filepath, 'rb') as f:
            # Load the data from the pickle file
            data = pickle.load(f)
        # Append loaded data to the list
        loaded_data.append(data)

len(loaded_data)

451

In [ ]:
import os
import pandas as pd

# Directory containing CSV files
directory = 'batch_results_sequences_Bert4rec_last_version/'

# Initialize an empty list to store DataFrames
df_list = []

# Loop through all the files in the directory
for filename in os.listdir(directory):
    if filename.endswith(".csv"):
        # Construct full file path
        file_path = os.path.join(directory, filename)

        # Read the CSV file into a DataFrame and append it to the list
        df = pd.read_csv(file_path)
        df_list.append(df)

# Concatenate all DataFrames in the list into a single DataFrame
final_df = pd.concat(df_list, ignore_index=True)

# Display or use final_df as needed
print(final_df.head())


   user_id predicted_sequence
0  1057489                [6]
1  1057491                [6]
2  1057493         [4, 13, 6]
3  1057502                [6]
4  1057507                [6]


In [ ]:
final_df

,user_id,predicted_sequence
0,1057489,[6]
1,1057491,[6]
2,1057493,"[4, 13, 6]"
3,1057502,[6]
4,1057507,[6]
...,...,...
19285,504,[]
19286,511,"[13, 13, 8, 8, 8, 6]"
19287,529,"[4, 1, 6]"
19288,530,"[8, 4, 6]"


In [ ]:
type(final_df['predicted_sequence'][0])

str

In [ ]:
import ast
final_df['predicted_sequence'] = final_df['predicted_sequence'].apply(ast.literal_eval)

In [ ]:
filtered_df = final_df[final_df['predicted_sequence'].apply(lambda x: x != [6])]
filtered_df

,user_id,predicted_sequence
2,1057493,"[4, 13, 6]"
13,1057553,"[4, 6]"
16,1057567,"[4, 13, 13, 13, 6]"
17,1057571,"[13, 6]"
18,1057576,"[4, 6]"
...,...,...
19285,504,[]
19286,511,"[13, 13, 8, 8, 8, 6]"
19287,529,"[4, 1, 6]"
19288,530,"[8, 4, 6]"


In [ ]:
filtered_df.columns


Index(['user_id', 'predicted_sequence'], dtype='object')

In [ ]:
filtered_df = filtered_df.reset_index(drop=True)

type(filtered_df['predicted_sequence'][0])

list

In [ ]:
unique_items = set(item for sublist in final_df['predicted_sequence'] for item in sublist)
unique_items

{1, 2, 3, 4, 5, 6, 7, 8, 10, 13, 16}

In [ ]:
items_ids = {
    '[PAD]': 0, 'CAT_1': 1, 'CAT_2': 2, 'CAT_3': 3, 'CAT_4': 4,
    'CAT_5': 5, 'CAT_0': 6, 'CAT_6': 7, 'CAT_7': 8, 'CAT_8': 9,
    'CAT_9': 10, 'CAT_10': 11, 'CAT_11': 12, 'CAT_12': 13,
    'CAT_13': 14, 'CAT_14': 15, 'CAT_15': 16, 'CAT_16': 17,
    'CAT_17': 18, 'CAT_18': 19, 'CAT_19': 20, 'CAT_20': 21
}

# Step 1: Reverse the items_ids dictionary to map IDs to genres
id_to_genre = {v: k for k, v in items_ids.items()}

# Step 2: Create a function to map the predicted sequence to genres
def map_sequence_to_genres(sequence):
    return [id_to_genre[item_id] for item_id in sequence]

# Step 3: Apply the mapping function to the 'predicted_sequence' column
filtered_df['_genres_mapped'] = filtered_df['predicted_sequence'].apply(map_sequence_to_genres)

# Step 4: Check the result
print(filtered_df.head())

   user_id  predicted_sequence                          _genres_mapped
0  1057493          [4, 13, 6]                  [CAT_4, CAT_12, CAT_0]
1  1057553              [4, 6]                          [CAT_4, CAT_0]
2  1057567  [4, 13, 13, 13, 6]  [CAT_4, CAT_12, CAT_12, CAT_12, CAT_0]
3  1057571             [13, 6]                         [CAT_12, CAT_0]
4  1057576              [4, 6]                          [CAT_4, CAT_0]


In [ ]:
filtered_df.to_csv('final_predictions_bert4rec.csv', index=False)

In [ ]:
unique_genres_mapped_dict = dict(zip(data['genres_mapped'], data['genres']))
unique_genres_mapped_dict

{'CAT_7': 'Action',
 'CAT_12': 'Drama',
 'CAT_6': 'Romance',
 'CAT_13': 'War',
 'CAT_0': '[SEP]',
 'CAT_4': 'Comedy',
 'CAT_14': 'Western',
 'CAT_1': 'Adventure',
 'CAT_2': 'Animation',
 'CAT_3': 'Children',
 'CAT_5': 'Fantasy',
 'CAT_15': 'Sci-Fi',
 'CAT_9': 'Thriller',
 'CAT_16': 'Musical',
 'CAT_8': 'Crime',
 'CAT_11': 'Horror',
 'CAT_17': 'Film-Noir',
 'CAT_10': 'Mystery',
 'CAT_19': 'Documentary',
 'CAT_18': 'IMAX',
 'CAT_20': '(no genres listed)'}